# Price Prediction Dataset EDA

This notebook visualizes correlations between the actual boundary variables and the standardized price target.

In [ ]:
from pathlib import Path
import os


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        dataset_path = candidate / "output/clean_datasets/price_prediction_dataset.csv"
        if dataset_path.exists():
            return candidate
    raise FileNotFoundError("Could not find output/clean_datasets/price_prediction_dataset.csv")


PROJECT_ROOT = find_project_root()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

INPUT_PATH = PROJECT_ROOT / "output/clean_datasets/price_prediction_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "output/EDA_figures/price_prediction"
METHOD = "pearson"

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["font.size"] = 10

print(f"Project root: {PROJECT_ROOT}")
print(f"Input dataset: {INPUT_PATH}")

## Load Dataset

In [ ]:
df = pd.read_csv(INPUT_PATH, parse_dates=["time"])
numeric = df.select_dtypes(include="number")

print(f"Rows: {len(df):,}")
print(f"Numeric columns: {len(numeric.columns)}")
display(df.head())

## Correlation Heatmap

In [ ]:
corr = numeric.corr(method=METHOD)


def plot_correlation_heatmap(corr: pd.DataFrame, method: str = "pearson"):
    labels = corr.columns.tolist()
    fig, ax = plt.subplots(figsize=(10, 8))
    norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    image = ax.imshow(corr, cmap="coolwarm", norm=norm)

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_title(f"{method.title()} Correlation Heatmap")

    for row_idx, row_label in enumerate(labels):
        for col_idx, col_label in enumerate(labels):
            value = corr.loc[row_label, col_label]
            text_color = "white" if abs(value) > 0.65 else "black"
            ax.text(
                col_idx,
                row_idx,
                f"{value:.2f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=9,
            )

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Correlation")
    fig.tight_layout()
    return fig, ax


heatmap_fig, heatmap_ax = plot_correlation_heatmap(corr, METHOD)
display(heatmap_fig)

## Correlation With Price

In [ ]:
price_corr = corr["price"].drop("price").sort_values(key=lambda values: values.abs(), ascending=True)
colors = ["#b2182b" if value >= 0 else "#2166ac" for value in price_corr]

bar_fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(price_corr.index, price_corr.values, color=colors)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_xlim(-1, 1)
ax.set_xlabel("Correlation with price")
ax.set_title(f"{METHOD.title()} Correlation With Price")

for y_pos, value in enumerate(price_corr.values):
    label_x = value + 0.03 if value >= 0 else value - 0.03
    ha = "left" if value >= 0 else "right"
    ax.text(label_x, y_pos, f"{value:.2f}", va="center", ha=ha)

bar_fig.tight_layout()
display(bar_fig)

## Save Figures

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

matrix_path = OUTPUT_DIR / "EDA_price_prediction_correlation_matrix.csv"
heatmap_path = OUTPUT_DIR / "EDA_price_prediction_heatmap.png"
price_corr_path = OUTPUT_DIR / "EDA_price_prediction_price_correlations.png"

corr.to_csv(matrix_path)
heatmap_fig.savefig(heatmap_path, bbox_inches="tight")
bar_fig.savefig(price_corr_path, bbox_inches="tight")

print(f"Saved correlation matrix: {matrix_path}")
print(f"Saved heatmap: {heatmap_path}")
print(f"Saved price-correlation chart: {price_corr_path}")